# Amodal fine-tuning trên Kaggle GPU

Trước khi chạy:
1. Settings → Accelerator → **GPU T4 x2** (hoặc P100).
2. Add-ons → Secrets → thêm `OPENAI_API_KEY` (và `HF_TOKEN` nếu base model bị gated).
3. Add Data → upload dataset ảnh gốc của bạn (ít ảnh) dưới dạng Kaggle Dataset riêng.

In [ ]:
!git clone https://github.com/<user>/<repo>.git
%cd <repo>
!pip install -q -r requirements.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")
# os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")  # nếu cần

In [ ]:
# Copy ảnh từ Kaggle Dataset đã add vào data/raw
!mkdir -p data/raw
!cp -r /kaggle/input/<ten-dataset-cua-ban>/* data/raw/
!ls data/raw | head

In [ ]:
# (Tuỳ chọn) tải mẫu COCOA để có ground truth eval
# Cần annotation tar.gz đã xin quyền tải thủ công, upload như một Kaggle Dataset riêng
# !python scripts/01_prepare_cocoa.py --annotation_targz /kaggle/input/cocoa-ann/cocoa_annotation.tar.gz --out data/cocoa --n_samples 50

In [ ]:
!python scripts/02_pseudo_label.py --images data/raw --out data/pseudo_labels --query "the main object"

In [ ]:
!python scripts/03_prepare_training_data.py --pseudo data/pseudo_labels --out data/train_ready

In [ ]:
!python scripts/04_train_lora.py --config configs/config.yaml

In [ ]:
# Cần data/cocoa (từ cell tuỳ chọn phía trên) để có ground truth eval
!python scripts/05_evaluate.py --config configs/config.yaml --lora_path outputs/lora/final

In [ ]:
# Kaggle không giữ session lâu dài -> nén checkpoint để tải về
!zip -r lora_checkpoint.zip outputs/lora/final